# Consulta de reseñas

En este notebook, se ejecuta paso a paso todo lo necesario para realizar la consulta del conteo de reseñas de cada tipo (Buena, Mala, Informativa) para cada alojamiento, imprimiendo el top 5.

Para esta consulta, es necesario aplicar un modelo de PLN. Nosotros usaremos uno muy básico de análisis de sentimiento, que clasifica textos según palabras clave, asociando sentimientos buenos o malos a las palabras. Este modelo es **VADER**.

## Importación de librerías

In [1]:
import sys
import pathlib

notebook_dir = pathlib.Path.cwd()
project_root = notebook_dir.parent.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Directorio raíz añadido al path: {project_root}")

Directorio raíz añadido al path: /home/dani/Universidad/3o/2o-cuatri/SDPDII/Practicas/proy_SSDD_II


In [2]:
from src.kafka.consumer_kafka import create_kafka_stream_df, consumer_kafka_avro

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, pandas_udf, window
from pyspark.sql.types import StringType
import pandas as pd

import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

## Verificar cola de Apache Kafka

El siguiente paso es comprobar que están los datos esperando en la cola de Apache Kafka. Para ello, vamos a usar la función implementada de consumidor, que imprime por pantalla los datos recibidos.

In [3]:
consumer_kafka_avro(
    topic='airbnb_reviews_gold', 
    idle_timeout_seconds=10, 
    log_to_file=False
)

2026-05-03 19:44:56,323 [INFO] Escritura de logs en archivo desactivada.
2026-05-03 19:44:56,324 [INFO] --- INICIANDO MONITOR PARA: airbnb_reviews_gold ---
2026-05-03 19:45:06,365 [INFO] Timeout alcanzado: 10s sin mensajes. Cerrando...
2026-05-03 19:45:06,372 [INFO] ============================================================
2026-05-03 19:45:06,373 [INFO] RESUMEN FINAL - TÓPICO: airbnb_reviews_gold
2026-05-03 19:45:06,374 [INFO] - Total mensajes: 0
2026-05-03 19:45:06,375 [INFO] - Columnas: 0
2026-05-03 19:45:06,377 [INFO] ============================================================


## Consulta con Spark Streaming

Primero, tenemos que crear la sesión de Spark. Es importante destacar el paquete de Spark para Apache Kafka y Avro.

In [4]:
spark = SparkSession.builder \
    .appName("Notebook_Borrador_Reviews") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1,org.apache.spark:spark-avro_2.13:4.1.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Session iniciada con éxito.")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 19:45:07 WARN Utils: Your hostname, dani, resolves to a loopback address: 127.0.1.1; using 10.20.128.138 instead (on interface gpd0)
26/05/03 19:45:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/dani/Universidad/3o/2o-cuatri/SDPDII/Practicas/proy_SSDD_II/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/dani/.ivy2.5.2/cache
The jars for the packages stored in: /home/dani/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
org.apache.spark#spark-avro_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-98fdd61e-fd8e-4f1a-b6ff-519bd3d31bbf;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.1.1 in central
	found org.apache.spark#spark-token-provider-kafk

Spark Session iniciada con éxito.


Definimos la función a usar con PLN (**VADER**). Para esta función, usaremos la librería de NLTK (vista en clase).

In [5]:
# Definimos el modelo PLN
@pandas_udf(StringType())
def clasificar_review(comentarios: pd.Series) -> pd.Series:
    nltk.download('vader_lexicon', quiet=True)
    
    # Inicializamos el analizador de VADER
    sia = SentimentIntensityAnalyzer()
    
    def evaluar_sentimiento(texto):
        if not texto or pd.isna(texto):
            return "informativa"
        
        # VADER evalúa el texto y devuelve algo como: 
        # {'neg': 0.0, 'neu': 0.45, 'pos': 0.55, 'compound': 0.83}
        scores = sia.polarity_scores(str(texto))
        
        # Nos quedamos con el valor resumen ('compound')
        compound = scores['compound']
        
        # Aplicamos nuestros umbrales
        if compound > 0.15: 
            return "buena"
        elif compound < -0.15: 
            return "mala"
        else: 
            return "informativa"
            
    return comentarios.apply(evaluar_sentimiento)

Antes de realizar el flujo de verdad, vamos a comprobar el funcionamiento del modelo PLN usado con unas reviews de prueba sencillas.

In [6]:
# --- PRUEBA UNITARIA DEL MODELO VADER ---

# 1. Definimos casos extremos y claros para volver "loco" al modelo
datos_prueba = [
    ("1", "The apartment is amazing, wonderful views and very clean. We loved it!"),  # Buena
    ("2", "Horrible experience. It was noisy, dirty and the host didn't reply."), # Mala
    ("3", "The apartment has a microwave, two single beds and is on the 3rd floor."), # Informativa
    ("4", None)
]

# 2. Creamos un DataFrame estático temporal de Spark
df_test = spark.createDataFrame(datos_prueba, ["id_falso", "comments"])

print("Evaluando textos de prueba controlados...")

# 3. Le aplicamos tu función UDF exactamente igual que en el stream
df_resultado_test = df_test.withColumn("clasificacion_vader", clasificar_review(col("comments")))

# 4. Mostramos el resultado
df_resultado_test.select("comments", "clasificacion_vader").show(truncate=False)

Evaluando textos de prueba controlados...


+-----------------------------------------------------------------------+-------------------+
|comments                                                               |clasificacion_vader|
+-----------------------------------------------------------------------+-------------------+
|The apartment is amazing, wonderful views and very clean. We loved it! |buena              |
|Horrible experience. It was noisy, dirty and the host didn't reply.    |mala               |
|The apartment has a microwave, two single beds and is on the 3rd floor.|informativa        |
|NULL                                                                   |informativa        |
+-----------------------------------------------------------------------+-------------------+



Comenzamos el flujo de Spark Streaming. Primero leemos los datos de la cola, aplicamos la clasificación de reviews y el Event Time. Por último agrupamos por tiempo de evento, anuncio (`listing_id`) y el tipo de review (obtenido en el paso anterior).

In [7]:
# 1. Leemos el stream usando TU función importada
df_reviews_crudo = create_kafka_stream_df(spark, "airbnb_reviews_gold")

# 2. Aplicamos clasificación y cast de fecha
df_procesado = df_reviews_crudo \
    .withColumn("tipo_review", clasificar_review(col("comments"))) \
    .withColumn("event_timestamp", col("date").cast("timestamp"))

# 3. Ventanas y conteo
df_agrupado = df_procesado \
    .withWatermark("event_timestamp", "7 days") \
    .groupBy(
        window(col("event_timestamp"), "365 days"),
        col("tipo_review")
    ).count()

# Ordenamos cronológicamente para ver la evolución
df_final = df_agrupado.orderBy(col("window.start").desc(), col("tipo_review"))

Iniciamos el flujo streaming.

In [8]:
# Celda 4: Iniciar stream
query_reviews = df_final.writeStream \
    .outputMode("complete") \
    .format("memory") \
    .queryName("tabla_resultados_reviews") \
    .start()

26/05/03 19:45:22 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-87a20e7c-2042-4d4b-83b6-e0cd20402d02. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/03 19:45:22 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [19]:
# Celda 5: Consultar
spark.sql("SELECT * FROM tabla_resultados_reviews").show(truncate=False)

+------------------------------------------+-----------+-----+
|window                                    |tipo_review|count|
+------------------------------------------+-----------+-----+
|{2024-12-18 01:00:00, 2025-12-18 01:00:00}|buena      |54586|
|{2024-12-18 01:00:00, 2025-12-18 01:00:00}|informativa|33214|
|{2024-12-18 01:00:00, 2025-12-18 01:00:00}|mala       |13164|
|{2023-12-19 01:00:00, 2024-12-18 01:00:00}|buena      |61290|
|{2023-12-19 01:00:00, 2024-12-18 01:00:00}|informativa|34270|
|{2023-12-19 01:00:00, 2024-12-18 01:00:00}|mala       |13794|
|{2022-12-19 01:00:00, 2023-12-19 01:00:00}|buena      |50280|
|{2022-12-19 01:00:00, 2023-12-19 01:00:00}|informativa|24925|
|{2022-12-19 01:00:00, 2023-12-19 01:00:00}|mala       |10287|
|{2021-12-19 01:00:00, 2022-12-19 01:00:00}|buena      |39111|
|{2021-12-19 01:00:00, 2022-12-19 01:00:00}|informativa|15833|
|{2021-12-19 01:00:00, 2022-12-19 01:00:00}|mala       |5751 |
|{2020-12-19 01:00:00, 2021-12-19 01:00:00}|buena      